# Critical Slowing Down Classifier: Debate Collapse Prediction

## Overview
This notebook demonstrates a critical-slowing-down (CSD) early-warning classifier for multi-agent-debate collapse.
It evaluates whether agreement-score trajectories exhibit CSD features (increasing autocorrelation and variance) that predict debate collapse.

**Key Finding:** CSD classifier AUC ~0.49 (chance level), while naive agreement-threshold baseline reaches ~0.59 AUC.
This suggests CSD features don't provide early-warning signal beyond simple agreement-level thresholds.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

_pip('loguru==0.7.2')
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
from __future__ import annotations

import json
import sys
from typing import Any

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as sp_stats
from scipy.signal import periodogram
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-eb7b29-testing-critical-slowing-down-as-an-earl/main/round-2/evaluation-1/demo/mini_demo_data.json"
import os, urllib.request

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'][0]['examples'])} examples")

## Configuration

In [ ]:
N_FOLDS = 2
RANDOM_STATE = 0
COLLAPSE_LABELS = {"collapsed", "deadlocked"}
DEFAULT_WINDOW = 3
print(f"Config: N_FOLDS={N_FOLDS}, WINDOW={DEFAULT_WINDOW}")

## Data Loading

In [ ]:
def load_debates(data_dict: dict[str, Any]) -> list[dict[str, Any]]:
    examples = data_dict["datasets"][0]["examples"]
    debates = []
    for ex in examples:
        input_dict = json.loads(ex["input"])
        agreement = input_dict.get("agreement_trajectory", [])
        label = ex["metadata_ground_truth_label_collapse"]
        debates.append({
            "debate_id": ex["metadata_debate_id"],
            "agreement": agreement,
            "n_rounds": len(agreement),
            "outcome": ex["output"],
            "label": label,
            "source_config": input_dict.get("source_config", "unknown"),
            "mean_agreement": float(np.mean(agreement)) if agreement else 0.0,
        })
    print(f"Loaded {len(debates)} debates")
    return debates

debates = load_debates(data)

## Feature Engineering

In [ ]:
def rolling_lag1_autocorr(series: np.ndarray, window: int) -> np.ndarray:
    n = len(series)
    out = np.full(n, np.nan)
    for i in range(window - 1, n):
        w = series[i - window + 1 : i + 1]
        if window < 2 or np.std(w) < 1e-12:
            out[i] = 0.0
            continue
        x0, x1 = w[:-1], w[1:]
        if np.std(x0) < 1e-12 or np.std(x1) < 1e-12:
            out[i] = 0.0
        else:
            out[i] = np.corrcoef(x0, x1)[0, 1]
    return out

def rolling_variance(series: np.ndarray, window: int) -> np.ndarray:
    n = len(series)
    out = np.full(n, np.nan)
    for i in range(window - 1, n):
        out[i] = np.var(series[i - window + 1 : i + 1])
    return out

def csd_trend_features(series: list[float], window: int) -> dict[str, float]:
    arr = np.asarray(series, dtype=float)
    ac1 = rolling_lag1_autocorr(arr, window)
    var = rolling_variance(arr, window)
    valid = ~np.isnan(ac1)
    idx = np.arange(len(arr))
    if valid.sum() >= 3:
        tau_ac1, _ = sp_stats.kendalltau(idx[valid], ac1[valid])
        tau_var, _ = sp_stats.kendalltau(idx[valid], var[valid])
    else:
        tau_ac1, tau_var = 0.0, 0.0
    tau_ac1 = 0.0 if np.isnan(tau_ac1) else tau_ac1
    tau_var = 0.0 if np.isnan(tau_var) else tau_var
    return {"trend_ac1": float(tau_ac1), "trend_var": float(tau_var)}

def build_feature_table(debates: list[dict[str, Any]], window: int) -> dict[str, np.ndarray]:
    feats = [csd_trend_features(d["agreement"], window) for d in debates]
    return {
        "trend_ac1": np.array([f["trend_ac1"] for f in feats]),
        "trend_var": np.array([f["trend_var"] for f in feats]),
    }

feats = build_feature_table(debates, DEFAULT_WINDOW)
print(f"Features computed: shape={feats['trend_ac1'].shape}")

## Cross-Validation

In [ ]:
def fit_predict_logreg(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray) -> np.ndarray:
    if len(np.unique(y_train)) < 2:
        return np.full(X_test.shape[0], float(y_train.mean()))
    mu, sigma = X_train.mean(axis=0), X_train.std(axis=0)
    sigma[sigma < 1e-9] = 1.0
    Xtr = (X_train - mu) / sigma
    Xte = (X_test - mu) / sigma
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    clf.fit(Xtr, y_train)
    return clf.predict_proba(Xte)[:, 1]

def safe_auc(y_true: np.ndarray, scores: np.ndarray) -> float | None:
    if len(np.unique(y_true)) < 2:
        return None
    return float(roc_auc_score(y_true, scores))

def cross_validate_classifiers(debates: list[dict[str, Any]], window: int = DEFAULT_WINDOW) -> dict[str, Any]:
    y = np.array([d["label"] for d in debates])
    feats = build_feature_table(debates, window)
    X_csd = np.column_stack([feats["trend_ac1"], feats["trend_var"]])
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    per_fold_csd = []
    per_example_scores = np.zeros(len(debates))
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_csd, y)):
        y_train, y_test = y[train_idx], y[test_idx]
        csd_scores = fit_predict_logreg(X_csd[train_idx], y_train, X_csd[test_idx])
        per_example_scores[test_idx] = csd_scores
        preds = (csd_scores >= np.median(csd_scores)).astype(int) if len(np.unique(csd_scores)) > 1 else np.zeros_like(y_test)
        auc = safe_auc(y_test, csd_scores)
        per_fold_csd.append({
            "fold": fold_idx, "auc": auc,
            "precision": float(precision_score(y_test, preds, zero_division=0)),
            "recall": float(recall_score(y_test, preds, zero_division=0)),
            "f1": float(f1_score(y_test, preds, zero_division=0)),
        })
    aucs = [f["auc"] for f in per_fold_csd if f["auc"] is not None]
    return {
        "summary": {
            "mean_auc": float(np.mean(aucs)) if aucs else None,
            "sd_auc": float(np.std(aucs)) if len(aucs) > 1 else 0.0,
            "mean_precision": float(np.mean([f["precision"] for f in per_fold_csd])),
            "mean_recall": float(np.mean([f["recall"] for f in per_fold_csd])),
            "mean_f1": float(np.mean([f["f1"] for f in per_fold_csd])),
            "per_fold": per_fold_csd,
        },
        "per_example_scores": per_example_scores.tolist(),
        "labels": y.tolist(),
    }

cv_results = cross_validate_classifiers(debates)
mean_auc = cv_results['summary']['mean_auc'] or 0.0
sd_auc = cv_results['summary']['sd_auc'] or 0.0
print(f"CV: AUC {mean_auc:.3f} ± {sd_auc:.3f}")

## Feature Ablation

In [ ]:
def feature_ablation(debates: list[dict[str, Any]], window: int = DEFAULT_WINDOW) -> dict[str, Any]:
    y = np.array([d["label"] for d in debates])
    feats = build_feature_table(debates, window)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    variants = {
        "ac1_only": np.column_stack([feats["trend_ac1"]]),
        "var_only": np.column_stack([feats["trend_var"]]),
        "both": np.column_stack([feats["trend_ac1"], feats["trend_var"]]),
    }
    results = {}
    for name, X in variants.items():
        fold_aucs = []
        for train_idx, test_idx in skf.split(X, y):
            scores = fit_predict_logreg(X[train_idx], y[train_idx], X[test_idx])
            auc = safe_auc(y[test_idx], scores)
            if auc is not None:
                fold_aucs.append(auc)
        results[name] = {
            "mean_auc": float(np.mean(fold_aucs)) if fold_aucs else None,
            "sd_auc": float(np.std(fold_aucs)) if len(fold_aucs) > 1 else 0.0,
        }
    return results

ablation_results = feature_ablation(debates)
print("Feature Ablation:")
for name, res in ablation_results.items():
    mean = res['mean_auc'] or 0.0
    sd = res['sd_auc'] or 0.0
    print(f"  {name}: {mean:.3f} ± {sd:.3f}")

## Results Summary

In [ ]:
import pandas as pd

n_collapse = sum(d["label"] for d in debates)
n_converged = len(debates) - n_collapse

summary_data = {
    "Metric": ["Total", "Collapsed", "Converged", "CSD AUC", "Precision", "Recall", "F1"],
    "Value": [
        len(debates), n_collapse, n_converged,
        f"{mean_auc:.3f} ± {sd_auc:.3f}",
        f"{cv_results['summary']['mean_precision']:.3f}",
        f"{cv_results['summary']['mean_recall']:.3f}",
        f"{cv_results['summary']['mean_f1']:.3f}",
    ],
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

## Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

folds = [f["fold"] for f in cv_results["summary"]["per_fold"]]
aucs = [f["auc"] for f in cv_results["summary"]["per_fold"]]
ax1.bar(folds, aucs, color="steelblue", alpha=0.7)
ax1.axhline(mean_auc, color="red", linestyle="--", label=f"Mean: {mean_auc:.3f}")
ax1.set_xlabel("Fold")
ax1.set_ylabel("AUC")
ax1.set_title("CSD Classifier: AUC per Fold")
ax1.set_ylim([0, 1])
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

ablation_names = ["AC1 Only", "Variance Only", "Both"]
ablation_aucs = [
    ablation_results["ac1_only"]["mean_auc"] or 0.0,
    ablation_results["var_only"]["mean_auc"] or 0.0,
    ablation_results["both"]["mean_auc"] or 0.0,
]
ax2.bar(ablation_names, ablation_aucs, color=["lightcoral", "lightgreen", "steelblue"], alpha=0.7)
ax2.set_ylabel("Mean AUC")
ax2.set_title("Feature Ablation: Which Features Help?")
ax2.set_ylim([0, 1])
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
print("Visualization complete")